# ======================================================================
# MODULE 2: ChaCha20-Poly1305 Authenticated Encryption
# ======================================================================
#
# **Project:** QVSC — Quantum-assisted Video Steganographic Communication
#
# **Purpose:** Encrypt the secret message (text or image) using the
# quantum-derived 256-bit key from Module 1 (E91 + SHA3-256).
# ChaCha20-Poly1305 AEAD provides both confidentiality and integrity.
#
# **Input:**
# - `quantum_key.bin` — 256-bit symmetric key from Module 1
# - Secret message (text string or image file)
#
# **Output:**
# - `ciphertext.bin` — Binary file containing: Nonce (12B) || Ciphertext || Tag (16B)
# - This binary blob will be converted to a bitstream and embedded by Module 4 (DCT-QIM)
#
# **Algorithm:**
# ```
# 1. Load 256-bit key K from quantum_key.bin
# 2. Read secret message M (text → UTF-8 bytes, or image → raw bytes)
# 3. Generate random 96-bit (12-byte) nonce
# 4. Ciphertext C, Tag T = ChaCha20_Poly1305_Encrypt(K, Nonce, M, AAD)
# 5. Output = Nonce || Ciphertext || Tag
# ```
#
# **Tools:** PyCryptodome (Crypto.Cipher.ChaCha20_Poly1305)
#
# ======================================================================

## Section 1: Library Imports

In [1]:
import os
import numpy as np
from Crypto.Cipher import ChaCha20_Poly1305
from Crypto.Hash import SHA3_256
from Crypto.Random import get_random_bytes
import binascii
import struct

print("All libraries imported successfully.")

All libraries imported successfully.


## Section 2: Load Quantum-Derived Key from Module 1

The 256-bit (32-byte) key was generated by Module 1 using the E91 QKD protocol
followed by SHA3-256 hashing. It is stored in `quantum_key.bin`.

In [2]:
# ── Load the 256-bit quantum-derived key ──────────────────────────────
KEY_FILE = 'quantum_key.bin'

with open(KEY_FILE, 'rb') as f:
    key = f.read()

assert len(key) == 32, f"Key must be 32 bytes (256 bits), got {len(key)} bytes"

print(f"Key loaded from '{KEY_FILE}'")
print(f"Key (hex): {key.hex()}")
print(f"Key length: {len(key) * 8} bits")

Key loaded from 'quantum_key.bin'
Key (hex): 33949916729f9fa64d91d188d7e367ede4c020dbf3871400689f3d702e94d22d
Key length: 256 bits


## Section 3: Prepare the Secret Message

Module 2 supports two types of secret data:
- **Text message:** Encoded as UTF-8 bytes
- **Image file:** Read as raw binary bytes

A 4-byte header is prepended to identify the type:
- `0x00000001` → Text
- `0x00000002` → Image

This header allows Module 6 (decryption) to correctly reconstruct the original data.

In [3]:
# # ── Option A: Text message ────────────────────────────────────────────
# secret_text = "This is a top-secret message encrypted with a quantum-derived key using ChaCha20-Poly1305 AEAD. -- Hasibul Hasan . Roll : 2003127"

# # Encode text to bytes with a type header
# TEXT_TYPE_HEADER = struct.pack('>I', 1)   # 4 bytes: 0x00000001
# text_bytes = secret_text.encode('utf-8')
# plaintext = TEXT_TYPE_HEADER + text_bytes

# print(f"Secret message: '{secret_text}'")
# print(f"Plaintext length: {len(plaintext)} bytes ({len(plaintext) * 8} bits)")
# print(f"  (4-byte type header + {len(text_bytes)}-byte message)")

In [4]:
# ── Option B: Image file (uncomment to use instead of text) ──────────
IMAGE_TYPE_HEADER = struct.pack('>I', 2)   # 4 bytes: 0x00000002
image_path = 'secret_image.jpg'
with open(image_path, 'rb') as f:
    image_bytes = f.read()
plaintext = IMAGE_TYPE_HEADER + image_bytes
print(f"Image loaded: '{image_path}', size: {len(image_bytes)} bytes")
print(f"Total plaintext: {len(plaintext)} bytes")

Image loaded: 'secret_image.jpg', size: 74417 bytes
Total plaintext: 74421 bytes


## Section 4: ChaCha20-Poly1305 Encryption

**ChaCha20-Poly1305 AEAD** combines:
- **ChaCha20** stream cipher for encryption (confidentiality)
- **Poly1305** MAC for authentication (integrity)

The 256-bit key and a random 96-bit nonce are used to generate the keystream.
The keystream is XORed with the plaintext to produce the ciphertext.
Poly1305 generates a 128-bit (16-byte) authentication tag.

**Associated Data (AAD):** A header string that is authenticated but not encrypted.
This ensures the context of the message cannot be tampered with.

In [5]:
def encrypt_chacha20_poly1305(key, plaintext, aad=b"QVSC-Module2"):
    """
    Encrypt plaintext using ChaCha20-Poly1305 AEAD.
    
    Parameters:
        key       : bytes (32 bytes / 256 bits) — symmetric encryption key
        plaintext : bytes — the secret data to encrypt
        aad       : bytes — Associated Authenticated Data (authenticated, not encrypted)
    
    Returns:
        nonce      : bytes (12 bytes / 96 bits)
        ciphertext : bytes (same length as plaintext)
        tag        : bytes (16 bytes / 128 bits) — Poly1305 authentication tag
    """
    # Create cipher object — nonce is auto-generated (96-bit)
    cipher = ChaCha20_Poly1305.new(key=key)
    
    # Add Associated Authenticated Data
    cipher.update(aad)
    
    # Encrypt and generate authentication tag
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)
    
    # Retrieve the nonce used
    nonce = cipher.nonce
    
    return nonce, ciphertext, tag

print("Encryption function defined.")

Encryption function defined.


In [6]:
# ── Perform Encryption ────────────────────────────────────────────────
AAD = b"QVSC-Module2"  # Associated Authenticated Data

nonce, ciphertext, tag = encrypt_chacha20_poly1305(key, plaintext, AAD)

print("="*70)
print("  ENCRYPTION RESULT")
print("="*70)
print(f"  Nonce (hex):      {nonce.hex()}")
print(f"  Nonce length:     {len(nonce)} bytes ({len(nonce)*8} bits)")
print(f"  Ciphertext (hex): {ciphertext.hex()[:80]}...")
print(f"  Ciphertext len:   {len(ciphertext)} bytes ({len(ciphertext)*8} bits)")
print(f"  Tag (hex):        {tag.hex()}")
print(f"  Tag length:       {len(tag)} bytes ({len(tag)*8} bits)")
print("="*70)

  ENCRYPTION RESULT
  Nonce (hex):      a147f66eeab497894fa318e4
  Nonce length:     12 bytes (96 bits)
  Ciphertext (hex): f78c6d7fb8bc64b63bc521d864b66cc86ff0615409fff20438fdb75f07ba0fbfd0eda0c43ea08ab6...
  Ciphertext len:   74421 bytes (595368 bits)
  Tag (hex):        5b91f77f6f964b5debf46dc0ba4a26e0
  Tag length:       16 bytes (128 bits)


## Section 5: Save Encrypted Data to Binary File

The output binary file is structured as:
```
┌──────────────┬───────────────────────┬──────────────────┐
│  Nonce (12B) │  Ciphertext (N bytes) │   Tag (16B)      │
└──────────────┴───────────────────────┴──────────────────┘
```

This format matches the QSAC paper's `.bin` file structure:
- 96-bit nonce, followed by ciphertext, followed by 128-bit tag.

Module 4 (DCT-QIM Embedding) will read this file and convert it to a bitstream
for embedding into the cover video's Cr channel.

In [7]:
# ── Save to binary file ───────────────────────────────────────────────
OUTPUT_FILE = 'ciphertext.bin'

encrypted_blob = nonce + ciphertext + tag

with open(OUTPUT_FILE, 'wb') as f:
    f.write(encrypted_blob)

print(f"Encrypted data saved to '{OUTPUT_FILE}'")
print(f"  File size: {len(encrypted_blob)} bytes ({len(encrypted_blob)*8} bits)")
print(f"  Structure: Nonce({len(nonce)}B) + Ciphertext({len(ciphertext)}B) + Tag({len(tag)}B)")
print(f"  Total bits to embed in video: {len(encrypted_blob) * 8}")

Encrypted data saved to 'ciphertext.bin'
  File size: 74449 bytes (595592 bits)
  Structure: Nonce(12B) + Ciphertext(74421B) + Tag(16B)
  Total bits to embed in video: 595592


## Section 6: Decryption and Verification (Module 6 Preview)

This section demonstrates the decryption process that will be used by Module 6
on the receiver side. It verifies that:
1. The authentication tag matches (integrity check)
2. The decrypted plaintext matches the original message

In [8]:
def decrypt_chacha20_poly1305(key, nonce, ciphertext, tag, aad=b"QVSC-Module2"):
    """
    Decrypt ciphertext using ChaCha20-Poly1305 AEAD and verify the MAC tag.
    
    Parameters:
        key        : bytes (32 bytes) — same key used for encryption
        nonce      : bytes (12 bytes) — nonce from encryption
        ciphertext : bytes — encrypted data
        tag        : bytes (16 bytes) — Poly1305 authentication tag
        aad        : bytes — must match the AAD used during encryption
    
    Returns:
        plaintext  : bytes — decrypted data (or raises ValueError if tag fails)
    """
    cipher = ChaCha20_Poly1305.new(key=key, nonce=nonce)
    cipher.update(aad)
    
    try:
        plaintext = cipher.decrypt_and_verify(ciphertext, tag)
        return plaintext
    except ValueError:
        raise ValueError("MAC verification failed! Data may have been tampered with.")

print("Decryption function defined.")

Decryption function defined.


In [9]:
# ── Read from the saved binary file and decrypt ───────────────────────
with open(OUTPUT_FILE, 'rb') as f:
    data = f.read()

# Parse the binary structure: Nonce(12) || Ciphertext(N) || Tag(16)
recv_nonce      = data[:12]
recv_ciphertext = data[12:-16]
recv_tag        = data[-16:]

print(f"Read from '{OUTPUT_FILE}':")
print(f"  Nonce:      {recv_nonce.hex()}")
print(f"  Ciphertext: {len(recv_ciphertext)} bytes")
print(f"  Tag:        {recv_tag.hex()}")

Read from 'ciphertext.bin':
  Nonce:      a147f66eeab497894fa318e4
  Ciphertext: 74421 bytes
  Tag:        5b91f77f6f964b5debf46dc0ba4a26e0


In [10]:
# ── Decrypt and verify ────────────────────────────────────────────────
decrypted_data = decrypt_chacha20_poly1305(key, recv_nonce, recv_ciphertext, recv_tag, AAD)

# Parse the type header
msg_type = struct.unpack('>I', decrypted_data[:4])[0]
msg_body = decrypted_data[4:]

print("="*70)
print("  DECRYPTION RESULT")
print("="*70)
print(f"  ✓ MAC verification PASSED — data integrity confirmed.")

if msg_type == 1:
    recovered_text = msg_body.decode('utf-8')
    print(f"  Message type: Text")
    print(f"  Recovered:    '{recovered_text}'")
    assert recovered_text == secret_text, "MISMATCH! Decrypted text differs from original."
    print(f"  ✓ Message matches original — encryption/decryption verified.")
elif msg_type == 2:
    print(f"  Message type: Image")
    print(f"  Image size:   {len(msg_body)} bytes")
    # Optionally save: with open('recovered_image.png','wb') as f: f.write(msg_body)

print("="*70)

  DECRYPTION RESULT
  ✓ MAC verification PASSED — data integrity confirmed.
  Message type: Image
  Image size:   74417 bytes


## Section 7: Tampered Data Test (Integrity Verification)

Demonstrates that modifying even a single bit of the ciphertext causes
the Poly1305 MAC verification to fail — proving the integrity guarantee.

In [11]:
# ── Tamper with the ciphertext (flip one bit) ─────────────────────────
tampered_ciphertext = bytearray(recv_ciphertext)
tampered_ciphertext[0] ^= 0x01   # Flip the LSB of the first byte
tampered_ciphertext = bytes(tampered_ciphertext)

print("Attempting decryption with tampered ciphertext...")
try:
    decrypt_chacha20_poly1305(key, recv_nonce, tampered_ciphertext, recv_tag, AAD)
    print("  ✗ ERROR: Tampered data was accepted (this should NOT happen!)")
except ValueError as e:
    print(f"  ✓ {e}")
    print(f"  ✓ Poly1305 MAC correctly rejected the tampered data.")

Attempting decryption with tampered ciphertext...
  ✓ MAC verification failed! Data may have been tampered with.
  ✓ Poly1305 MAC correctly rejected the tampered data.


## Section 8: Wrong Key Test (Key Sensitivity)

Demonstrates that using a different key (even 1 bit different) fails decryption.

In [12]:
# ── Attempt decryption with a wrong key ───────────────────────────────
wrong_key = bytearray(key)
wrong_key[0] ^= 0x01    # Flip one bit in the key
wrong_key = bytes(wrong_key)

print(f"Original key (first 4B): {key[:4].hex()}")
print(f"Wrong key    (first 4B): {wrong_key[:4].hex()}")
print()
print("Attempting decryption with wrong key...")
try:
    decrypt_chacha20_poly1305(wrong_key, recv_nonce, recv_ciphertext, recv_tag, AAD)
    print("  ✗ ERROR: Wrong key was accepted (this should NOT happen!)")
except ValueError as e:
    print(f"  ✓ {e}")
    print(f"  ✓ System correctly rejected decryption with the wrong key.")

Original key (first 4B): 33949916
Wrong key    (first 4B): 32949916

Attempting decryption with wrong key...
  ✓ MAC verification failed! Data may have been tampered with.
  ✓ System correctly rejected decryption with the wrong key.


## Section 9: Ciphertext Bit Analysis

Shows the bitstream that will be passed to Module 4 (DCT-QIM Embedding).
Good encryption should produce near-uniform distribution of 0s and 1s.

In [13]:
# ── Convert ciphertext.bin to bitstream (preview for Module 4) ────────
with open(OUTPUT_FILE, 'rb') as f:
    blob = f.read()

# Convert bytes to bit string
bitstream = ''.join(format(byte, '08b') for byte in blob)

total_bits = len(bitstream)
ones_count = bitstream.count('1')
zeros_count = bitstream.count('0')

print("="*70)
print("  CIPHERTEXT BITSTREAM ANALYSIS")
print("="*70)
print(f"  Total bits:  {total_bits}")
print(f"  Zeros:       {zeros_count} ({zeros_count/total_bits*100:.1f}%)")
print(f"  Ones:        {ones_count} ({ones_count/total_bits*100:.1f}%)")
print(f"  Ratio (1/0): {ones_count/zeros_count:.4f} (ideal ≈ 1.0000)")
print(f"")
print(f"  First 64 bits: {bitstream[:64]}")
print(f"  Last  64 bits: {bitstream[-64:]}")
print("="*70)
print(f"\n  → These {total_bits} bits will be embedded into the cover video by Module 4.")

  CIPHERTEXT BITSTREAM ANALYSIS
  Total bits:  595592
  Zeros:       297580 (50.0%)
  Ones:        298012 (50.0%)
  Ratio (1/0): 1.0015 (ideal ≈ 1.0000)

  First 64 bits: 1010000101000111111101100110111011101010101101001001011110001001
  Last  64 bits: 1110101111110100011011011100000010111010010010100010011011100000

  → These 595592 bits will be embedded into the cover video by Module 4.


## Section 10: Entropy Calculation

Shannon entropy of the ciphertext bytes. Ideal encryption should yield
entropy close to 8.0 bits/byte (maximum for byte-level data).

In [14]:
# ── Calculate Shannon entropy of ciphertext ───────────────────────────
def shannon_entropy(data_bytes):
    """Calculate Shannon entropy in bits per byte."""
    byte_array = np.frombuffer(data_bytes, dtype=np.uint8)
    hist, _ = np.histogram(byte_array, bins=256, range=(0, 256))
    probabilities = hist / len(byte_array)
    probabilities = probabilities[probabilities > 0]
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

plaintext_entropy  = shannon_entropy(plaintext)
ciphertext_entropy = shannon_entropy(ciphertext)

print("="*70)
print("  ENTROPY ANALYSIS")
print("="*70)
print(f"  Plaintext entropy:  {plaintext_entropy:.4f} bits/byte")
print(f"  Ciphertext entropy: {ciphertext_entropy:.4f} bits/byte")
print(f"  Maximum possible:   8.0000 bits/byte")
print(f"")
if ciphertext_entropy > 7.5:
    print(f"  ✓ High entropy indicates strong encryption (near-random output).")
else:
    print(f"  ⚠ Entropy is lower than expected. This may be due to short message length.")
print("="*70)

  ENTROPY ANALYSIS
  Plaintext entropy:  7.9750 bits/byte
  Ciphertext entropy: 7.9977 bits/byte
  Maximum possible:   8.0000 bits/byte

  ✓ High entropy indicates strong encryption (near-random output).


## Section 11: Summary

Displays a complete summary of Module 2 outputs for pipeline verification.

In [15]:
# ── Correlation, UACI, and NSCR Analysis ─────────────────────────────
ct_bytes = np.frombuffer(ciphertext, dtype=np.uint8).astype(np.float64)
pt_bytes = np.frombuffer(plaintext, dtype=np.uint8).astype(np.float64)

# 1. Adjacent Byte Correlation (Ciphertext)
x = ct_bytes[:-1]
y = ct_bytes[1:]
mean_x, mean_y = np.mean(x), np.mean(y)
num = np.sum((x - mean_x) * (y - mean_y))
den = np.sqrt(np.sum((x - mean_x)**2) * np.sum((y - mean_y)**2))
ct_correlation = num / den if den != 0 else 0

# 2. Adjacent Byte Correlation (Plaintext, for comparison)
px = pt_bytes[:-1]
py = pt_bytes[1:]
mean_px, mean_py = np.mean(px), np.mean(py)
pt_num = np.sum((px - mean_px) * (py - mean_py))
pt_den = np.sqrt(np.sum((px - mean_px)**2) * np.sum((py - mean_py)**2))
pt_correlation = pt_num / pt_den if pt_den != 0 else 0

# 3. UACI (Unified Average Changing Intensity)
min_len = min(len(pt_bytes), len(ct_bytes))
uaci = np.mean(np.abs(pt_bytes[:min_len] - ct_bytes[:min_len]) / 255.0) * 100

# 4. NSCR (Number of Samples Changing Rate)
different_bytes = np.sum(pt_bytes[:min_len] != ct_bytes[:min_len])
nscr = (different_bytes / min_len) * 100

# ── Final Summary ─────────────────────────────────────────────────────
print("▓"*70)
print("  MODULE 2 COMPLETE — ChaCha20-Poly1305 Authenticated Encryption")
print("▓"*70)
print(f"")
print(f"  Input:")
print(f"    Key file:           {KEY_FILE} (256-bit, from Module 1)")
print(f"    Key (hex):          {key.hex()}")
print(f"    Secret message:     {len(plaintext)} bytes")
print(f"")
print(f"  Encryption Parameters:")
print(f"    Algorithm:          ChaCha20-Poly1305 AEAD")
print(f"    Key size:           256 bits")
print(f"    Nonce size:         96 bits (12 bytes)")
print(f"    Tag size:           128 bits (16 bytes)")
print(f"    AAD:                {AAD.decode()}")
print(f"")
print(f"  Output:")
print(f"    Output file:        {OUTPUT_FILE}")
print(f"    Nonce (hex):        {nonce.hex()}")
print(f"    Tag (hex):          {tag.hex()}")
print(f"    Ciphertext size:    {len(ciphertext)} bytes")
print(f"    Total file size:    {len(encrypted_blob)} bytes ({len(encrypted_blob)*8} bits)")
print(f"")
print(f"  Encryption Quality Metrics:")
print(f"    Ciphertext entropy:       {ciphertext_entropy:.4f} bits/byte (ideal = 8.0)")
print(f"    Plaintext correlation:    {pt_correlation:.10f}")
print(f"    Ciphertext correlation:   {ct_correlation:.10f} (ideal ≈ 0.0)")
print(f"    UACI:                     {uaci:.4f}% (ideal ≈ 33.46%)")
print(f"    NSCR:                     {nscr:.4f}% (ideal ≈ 100%)")
print(f"")
print(f"  Verification:")
print(f"    ✓ Decryption test:      PASSED")
print(f"    ✓ Tamper detection:      PASSED")
print(f"    ✓ Wrong key rejection:   PASSED")
print(f"")
print("="*70)
print(f"  MODULE 2 OUTPUT → '{OUTPUT_FILE}' ready for Module 3/4 (DCT-QIM Embedding)")
print("="*70)

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  MODULE 2 COMPLETE — ChaCha20-Poly1305 Authenticated Encryption
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Input:
    Key file:           quantum_key.bin (256-bit, from Module 1)
    Key (hex):          33949916729f9fa64d91d188d7e367ede4c020dbf3871400689f3d702e94d22d
    Secret message:     74421 bytes

  Encryption Parameters:
    Algorithm:          ChaCha20-Poly1305 AEAD
    Key size:           256 bits
    Nonce size:         96 bits (12 bytes)
    Tag size:           128 bits (16 bytes)
    AAD:                QVSC-Module2

  Output:
    Output file:        ciphertext.bin
    Nonce (hex):        a147f66eeab497894fa318e4
    Tag (hex):          5b91f77f6f964b5debf46dc0ba4a26e0
    Ciphertext size:    74421 bytes
    Total file size:    74449 bytes (595592 bits)

  Encryption Quality Metrics:
    Ciphertext entropy:       7.9977 bits/byte (ideal = 8.0)
    Plaintext correlation:   

In [16]:
# 

import fec_module as fec

print("Loaded FEC file:", fec.__file__)
print(f"RS configuration: RS({fec.RS_N},{fec.RS_K})")
print("Parity bytes:", fec.RS_PARITY)
print(
    "Correctable byte errors/codeword:",
    fec.RS_PARITY // 2
)

with open("ciphertext.bin", "rb") as f:
    ciphertext = f.read()

fec_payload = fec.fec_encode(ciphertext)

with open("fec_payload.bin", "wb") as f:
    f.write(fec_payload)

report = fec.fec_overhead(len(ciphertext))

print("\nFEC report:")
print(report)
print("Saved payload bytes:", len(fec_payload))



Loaded FEC file: d:\4-2\CSE 4000 - thesis\ICCIT\all mod code v2\fec_module.py
RS configuration: RS(255,191)
Parity bytes: 64
Correctable byte errors/codeword: 32

FEC report:
{'data_bytes': 74449, 'codewords': 390, 'fec_bytes': 99450, 'fec_bits': 795600, 'redundancy_pct': 25.1}
Saved payload bytes: 99450
